# Projeto 02 — Previsão de Churn de Clientes

## Contexto

Churn representa o cancelamento ou abandono de um serviço por parte de um cliente.

Empresas que oferecem serviços recorrentes, como telecomunicações, bancos, plataformas digitais e empresas de assinatura, procuram identificar antecipadamente clientes com maior risco de cancelamento.

Neste projeto, construiremos modelos de Machine Learning e Deep Learning capazes de estimar a probabilidade de churn de cada cliente.

## Objetivo

Prever a variável:

- `0`: cliente permanece;
- `1`: cliente cancela.

Este é um problema de **classificação binária**.

## Pergunta de negócio

> A partir das características do cliente, de seu contrato e dos serviços utilizados, é possível identificar clientes com maior risco de cancelamento?

## Modelos previstos

1. modelo baseline;
2. Regressão Logística;
3. Random Forest;
4. rede neural.

## Métricas

- accuracy;
- precision;
- recall;
- F1-score;
- matriz de confusão;
- ROC AUC.

## 1. Importação das bibliotecas e reprodutibilidade

Nesta etapa, importaremos as bibliotecas utilizadas na manipulação, preparação, visualização, treinamento e avaliação dos modelos.

Também definiremos uma seed para melhorar a reprodutibilidade dos experimentos.

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"Seed definida: {SEED}")

## 2. Carregamento do dataset

O dataset contém informações de clientes de uma empresa fictícia de telecomunicações.

Cada linha representa um cliente. As colunas registram características como tempo de permanência, tipo de contrato, forma de pagamento, valores cobrados e serviços utilizados.

A variável alvo será `Churn`.

In [ ]:
url = (
    "https://raw.githubusercontent.com/"
    "IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

df = pd.read_csv(url)

print("Dataset carregado com sucesso.")

## 3. Primeira inspeção

O método `head()` mostra as primeiras linhas do dataset e ajuda a verificar os nomes das colunas, o formato dos valores e possíveis problemas de leitura.

In [ ]:
df.head()

In [ ]:
print("Shape do dataset:", df.shape)

O `shape` segue o formato:

```text
(número de clientes, número de colunas)
```

In [ ]:
df.info()

## 4. Verificações iniciais

Vamos verificar valores ausentes, duplicatas e os nomes das colunas antes de iniciar o tratamento.

In [ ]:
print("Valores ausentes:")
print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
print("Duplicatas:", df.duplicated().sum())

In [ ]:
print("Colunas:")
for coluna in df.columns:
    print("-", coluna)

## 5. Análise da variável Churn

A distribuição percentual permite verificar se as classes estão desbalanceadas.

Como a acurácia pode ser enganosa em classes desbalanceadas, também utilizaremos recall, F1-score e ROC AUC.

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
df["Churn"].value_counts().plot(kind="bar")

plt.title("Distribuição da variável Churn")
plt.xlabel("Churn")
plt.ylabel("Quantidade de clientes")
plt.xticks(rotation=0)
plt.show()

## 6. Conversão da coluna TotalCharges

`TotalCharges` pode ser carregada como texto por conter espaços vazios. Vamos convertê-la para formato numérico.

O parâmetro `errors="coerce"` transforma valores inválidos em `NaN`.

In [ ]:
print("Tipo original de TotalCharges:")
print(df["TotalCharges"].dtype)

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Tipo após conversão:")
print(df["TotalCharges"].dtype)

print(
    "Valores ausentes após conversão:",
    df["TotalCharges"].isnull().sum()
)

## 7. Remoção do identificador

`customerID` é apenas um identificador único e não representa uma característica comportamental ou contratual.

In [ ]:
df = df.drop(columns=["customerID"])
print("Coluna customerID removida.")

## 8. Conversão da variável alvo

A variável alvo será convertida para:

- `No` → `0`;
- `Yes` → `1`.

In [ ]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

df["Churn"].value_counts()

## 9. Separação entre features e alvo

`X` conterá as características dos clientes e `y` conterá a variável que queremos prever.

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

## 10. Separação entre treino, validação e teste

Vamos utilizar aproximadamente:

- 70% para treinamento;
- 15% para validação;
- 15% para teste.

O parâmetro `stratify` preserva a proporção de churn nos três conjuntos.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=SEED,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    random_state=SEED,
    stratify=y_temp
)

print("Treino:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test.shape, y_test.shape)

In [ ]:
print("Churn no treino:", round(y_train.mean(), 4))
print("Churn na validação:", round(y_val.mean(), 4))
print("Churn no teste:", round(y_test.mean(), 4))

## 11. Identificação das colunas numéricas e categóricas

Esses dois grupos precisarão de tratamentos diferentes no pipeline.

In [ ]:
colunas_numericas = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

colunas_categoricas = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Colunas numéricas:")
print(colunas_numericas)

print("\nColunas categóricas:")
print(colunas_categoricas)

## 12. Pipeline de pré-processamento

### Variáveis numéricas

- preenchimento de valores ausentes com a mediana;
- padronização com `StandardScaler`.

### Variáveis categóricas

- preenchimento com o valor mais frequente;
- transformação com One-Hot Encoding.

O `ColumnTransformer` permite aplicar cada tratamento ao grupo correto.

In [ ]:
pipeline_numerico = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

pipeline_categorico = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessador = ColumnTransformer([
    (
        "numerico",
        pipeline_numerico,
        colunas_numericas
    ),
    (
        "categorico",
        pipeline_categorico,
        colunas_categoricas
    )
])

## 13. Modelo baseline

O `DummyClassifier` prevê sempre a classe mais frequente.

Ele será nossa referência inicial. Um modelo real deve superar esse baseline, especialmente em recall, F1-score e ROC AUC.

In [ ]:
modelo_baseline = Pipeline([
    (
        "preprocessador",
        preprocessador
    ),
    (
        "modelo",
        DummyClassifier(strategy="most_frequent")
    )
])

modelo_baseline.fit(
    X_train,
    y_train
)

In [ ]:
y_pred_baseline = modelo_baseline.predict(
    X_test
)

print(
    classification_report(
        y_test,
        y_pred_baseline,
        zero_division=0
    )
)

# 15. Regressão Logística

A Regressão Logística será nosso primeiro modelo real de classificação.

Apesar do nome, ela é utilizada para problemas de classificação binária.

O modelo calcula uma probabilidade entre 0 e 1 e, por padrão, utiliza o limiar de 0,5:

```text
probabilidade < 0,5 → classe 0
probabilidade ≥ 0,5 → classe 1
```

Neste projeto:

- `0` representa cliente que permanece;
- `1` representa cliente que cancela.

## Pipeline da Regressão Logística

Vamos reutilizar o mesmo pré-processador criado anteriormente.

O pipeline executará automaticamente:

1. tratamento das variáveis numéricas;
2. tratamento das variáveis categóricas;
3. One-Hot Encoding;
4. padronização;
5. treinamento da Regressão Logística.

O parâmetro `max_iter=1000` aumenta o número máximo de iterações para ajudar o algoritmo a convergir.

In [ ]:
modelo_logistico = Pipeline([
    (
        "preprocessador",
        preprocessador
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            random_state=SEED
        )
    )
])

modelo_logistico.fit(
    X_train,
    y_train
)

print("Regressão Logística treinada com sucesso.")

## Previsões e probabilidades

Utilizaremos dois métodos:

- `predict()`: retorna a classe prevista;
- `predict_proba()`: retorna a probabilidade estimada de cada classe.

A coluna de índice `1` representa a probabilidade de churn.

In [ ]:
y_pred_logistico = modelo_logistico.predict(
    X_test
)

y_proba_logistico = modelo_logistico.predict_proba(
    X_test
)[:, 1]

print("Primeiras classes previstas:")
print(y_pred_logistico[:10])

print("\nPrimeiras probabilidades de churn:")
print(
    np.round(
        y_proba_logistico[:10],
        4
    )
)

# 16. Avaliação da Regressão Logística

Vamos calcular:

- accuracy;
- precision;
- recall;
- F1-score;
- ROC AUC.

## Interpretação das métricas

### Accuracy

Proporção total de previsões corretas.

### Precision

Entre os clientes classificados como churn, quantos realmente cancelaram.

### Recall

Entre os clientes que realmente cancelaram, quantos foram identificados pelo modelo.

### F1-score

Média harmônica entre precision e recall.

### ROC AUC

Mede a capacidade do modelo de separar as duas classes considerando diferentes thresholds.

Em um projeto de retenção, o recall costuma ser especialmente importante, pois um falso negativo representa um cliente em risco que não foi identificado.

In [ ]:
accuracy_logistico = accuracy_score(
    y_test,
    y_pred_logistico
)

precision_logistico = precision_score(
    y_test,
    y_pred_logistico,
    zero_division=0
)

recall_logistico = recall_score(
    y_test,
    y_pred_logistico,
    zero_division=0
)

f1_logistico = f1_score(
    y_test,
    y_pred_logistico,
    zero_division=0
)

roc_auc_logistico = roc_auc_score(
    y_test,
    y_proba_logistico
)

print("===== REGRESSÃO LOGÍSTICA =====")
print(f"Accuracy:  {accuracy_logistico:.4f}")
print(f"Precision: {precision_logistico:.4f}")
print(f"Recall:    {recall_logistico:.4f}")
print(f"F1-score:  {f1_logistico:.4f}")
print(f"ROC AUC:   {roc_auc_logistico:.4f}")

## Relatório de classificação

O `classification_report` apresenta as métricas separadamente para as duas classes.

A classe `1` é a principal classe de interesse porque representa churn.

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_logistico,
        target_names=[
            "Permanece",
            "Churn"
        ],
        zero_division=0
    )
)

# 17. Matriz de confusão

A matriz de confusão organiza as previsões em quatro grupos:

| Situação | Significado |
|---|---|
| Verdadeiro negativo | cliente permaneceu e foi classificado corretamente |
| Falso positivo | cliente permaneceu, mas foi classificado como churn |
| Falso negativo | cliente cancelou, mas não foi identificado |
| Verdadeiro positivo | cliente cancelou e foi identificado corretamente |

Em churn, o falso negativo é especialmente relevante porque representa um cliente em risco que não receberia uma ação de retenção.

In [ ]:
matriz_logistica = confusion_matrix(
    y_test,
    y_pred_logistico
)

display_matriz = ConfusionMatrixDisplay(
    confusion_matrix=matriz_logistica,
    display_labels=[
        "Permanece",
        "Churn"
    ]
)

display_matriz.plot(
    values_format="d"
)

plt.title(
    "Matriz de Confusão — Regressão Logística"
)

plt.show()

In [ ]:
tn, fp, fn, tp = matriz_logistica.ravel()

print("Verdadeiros negativos:", tn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdadeiros positivos:", tp)

# 18. Curva ROC

A curva ROC compara:

- taxa de verdadeiros positivos;
- taxa de falsos positivos;

em diferentes thresholds.

Quanto mais a curva se aproxima do canto superior esquerdo, melhor é a capacidade de separação do modelo.

Uma ROC AUC próxima de:

- `0,5`: desempenho semelhante ao acaso;
- `1,0`: separação perfeita.

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_proba_logistico,
    name="Regressão Logística"
)

plt.title(
    "Curva ROC — Regressão Logística"
)

plt.show()

# 19. Métricas do modelo baseline

Agora calcularemos as métricas do baseline para permitir uma comparação direta com a Regressão Logística.

Como o baseline prevê apenas a classe majoritária, ele provavelmente terá recall igual a zero para churn.

In [ ]:
y_proba_baseline = modelo_baseline.predict_proba(
    X_test
)[:, 1]

accuracy_baseline = accuracy_score(
    y_test,
    y_pred_baseline
)

precision_baseline = precision_score(
    y_test,
    y_pred_baseline,
    zero_division=0
)

recall_baseline = recall_score(
    y_test,
    y_pred_baseline,
    zero_division=0
)

f1_baseline = f1_score(
    y_test,
    y_pred_baseline,
    zero_division=0
)

roc_auc_baseline = roc_auc_score(
    y_test,
    y_proba_baseline
)

print("===== MODELO BASELINE =====")
print(f"Accuracy:  {accuracy_baseline:.4f}")
print(f"Precision: {precision_baseline:.4f}")
print(f"Recall:    {recall_baseline:.4f}")
print(f"F1-score:  {f1_baseline:.4f}")
print(f"ROC AUC:   {roc_auc_baseline:.4f}")

# 20. Comparação: baseline vs. Regressão Logística

A tabela abaixo permite verificar se a Regressão Logística realmente adicionou capacidade preditiva.

O modelo real deve superar o baseline principalmente em:

- recall;
- F1-score;
- ROC AUC.

In [ ]:
comparacao_inicial = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regressão Logística"
    ],

    "Accuracy": [
        accuracy_baseline,
        accuracy_logistico
    ],

    "Precision": [
        precision_baseline,
        precision_logistico
    ],

    "Recall": [
        recall_baseline,
        recall_logistico
    ],

    "F1-score": [
        f1_baseline,
        f1_logistico
    ],

    "ROC AUC": [
        roc_auc_baseline,
        roc_auc_logistico
    ]
})

comparacao_inicial.round(4)

## Visualização da comparação

Acurácia não será o único critério.

O gráfico abaixo compara as métricas mais relevantes para a classe de churn.

In [ ]:
metricas_grafico = comparacao_inicial.set_index(
    "Modelo"
)[[
    "Precision",
    "Recall",
    "F1-score",
    "ROC AUC"
]]

metricas_grafico.T.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title(
    "Baseline vs. Regressão Logística"
)

plt.xlabel("Métrica")
plt.ylabel("Resultado")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Modelo")
plt.show()

# 21. Análise dos coeficientes da Regressão Logística

A Regressão Logística permite analisar os coeficientes associados às variáveis após o pré-processamento.

De forma simplificada:

- coeficientes positivos estão associados a maior probabilidade prevista de churn;
- coeficientes negativos estão associados a menor probabilidade prevista de churn.

Essa análise mostra associação com a previsão, não causalidade.

In [ ]:
nomes_features = modelo_logistico.named_steps[
    "preprocessador"
].get_feature_names_out()

coeficientes = modelo_logistico.named_steps[
    "modelo"
].coef_[0]

df_coeficientes = pd.DataFrame({
    "Feature": nomes_features,
    "Coeficiente": coeficientes
})

df_coeficientes["Coeficiente_Absoluto"] = (
    df_coeficientes["Coeficiente"].abs()
)

df_coeficientes = df_coeficientes.sort_values(
    by="Coeficiente_Absoluto",
    ascending=False
)

df_coeficientes.head(15)

## Variáveis associadas ao aumento da probabilidade de churn

In [ ]:
df_coeficientes.sort_values(
    by="Coeficiente",
    ascending=False
)[[
    "Feature",
    "Coeficiente"
]].head(10)

## Variáveis associadas à redução da probabilidade de churn

In [ ]:
df_coeficientes.sort_values(
    by="Coeficiente",
    ascending=True
)[[
    "Feature",
    "Coeficiente"
]].head(10)

# 23. Random Forest

O Random Forest é um modelo baseado em múltiplas árvores de decisão.

Cada árvore é treinada com uma amostra dos dados e utiliza subconjuntos aleatórios de variáveis.

Ao final, as árvores combinam suas previsões.

Esse modelo pode capturar:

- relações não lineares;
- interações entre variáveis;
- padrões mais complexos do que a Regressão Logística.

Por outro lado, ele pode apresentar overfitting se for configurado com árvores muito profundas.

## Pipeline do Random Forest

Vamos reutilizar o mesmo pré-processador.

Como o One-Hot Encoding já está incluído no pipeline, o modelo receberá as variáveis categóricas transformadas em colunas numéricas.

Usaremos inicialmente:

- `n_estimators=300`: quantidade de árvores;
- `max_depth=None`: árvores sem limite fixo de profundidade;
- `min_samples_split=2`;
- `min_samples_leaf=1`;
- `class_weight="balanced"`: dá mais peso à classe minoritária;
- `random_state=42`: melhora a reprodutibilidade.

In [ ]:
modelo_random_forest = Pipeline([
    (
        "preprocessador",
        preprocessador
    ),
    (
        "modelo",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )
    )
])

modelo_random_forest.fit(
    X_train,
    y_train
)

print("Random Forest treinado com sucesso.")

## Previsões e probabilidades do Random Forest

Assim como na Regressão Logística, utilizaremos:

- `predict()` para obter as classes;
- `predict_proba()` para obter as probabilidades de churn.

In [ ]:
y_pred_random_forest = modelo_random_forest.predict(
    X_test
)

y_proba_random_forest = modelo_random_forest.predict_proba(
    X_test
)[:, 1]

print("Primeiras classes previstas:")
print(y_pred_random_forest[:10])

print("\nPrimeiras probabilidades de churn:")
print(
    np.round(
        y_proba_random_forest[:10],
        4
    )
)

# 24. Avaliação do Random Forest

Vamos utilizar as mesmas métricas aplicadas aos modelos anteriores:

- accuracy;
- precision;
- recall;
- F1-score;
- ROC AUC.

Isso permite uma comparação justa.

In [ ]:
accuracy_random_forest = accuracy_score(
    y_test,
    y_pred_random_forest
)

precision_random_forest = precision_score(
    y_test,
    y_pred_random_forest,
    zero_division=0
)

recall_random_forest = recall_score(
    y_test,
    y_pred_random_forest,
    zero_division=0
)

f1_random_forest = f1_score(
    y_test,
    y_pred_random_forest,
    zero_division=0
)

roc_auc_random_forest = roc_auc_score(
    y_test,
    y_proba_random_forest
)

print("===== RANDOM FOREST =====")
print(f"Accuracy:  {accuracy_random_forest:.4f}")
print(f"Precision: {precision_random_forest:.4f}")
print(f"Recall:    {recall_random_forest:.4f}")
print(f"F1-score:  {f1_random_forest:.4f}")
print(f"ROC AUC:   {roc_auc_random_forest:.4f}")

## Relatório de classificação

A classe `1` continua sendo a principal classe de interesse, pois representa churn.

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_random_forest,
        target_names=[
            "Permanece",
            "Churn"
        ],
        zero_division=0
    )
)

# 25. Matriz de confusão do Random Forest

A matriz de confusão permite verificar quantos clientes foram corretamente ou incorretamente classificados.

Em churn, devemos observar especialmente os falsos negativos.

In [ ]:
matriz_random_forest = confusion_matrix(
    y_test,
    y_pred_random_forest
)

display_matriz_rf = ConfusionMatrixDisplay(
    confusion_matrix=matriz_random_forest,
    display_labels=[
        "Permanece",
        "Churn"
    ]
)

display_matriz_rf.plot(
    values_format="d"
)

plt.title(
    "Matriz de Confusão — Random Forest"
)

plt.show()

In [ ]:
tn_rf, fp_rf, fn_rf, tp_rf = matriz_random_forest.ravel()

print("Verdadeiros negativos:", tn_rf)
print("Falsos positivos:", fp_rf)
print("Falsos negativos:", fn_rf)
print("Verdadeiros positivos:", tp_rf)

# 26. Curva ROC do Random Forest

A curva ROC mostra o desempenho do modelo em diferentes thresholds.

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_proba_random_forest,
    name="Random Forest"
)

plt.title(
    "Curva ROC — Random Forest"
)

plt.show()

# 27. Comparação entre os três modelos

Agora vamos comparar:

1. baseline;
2. Regressão Logística;
3. Random Forest.

As métricas serão reunidas em uma única tabela.

In [ ]:
comparacao_modelos = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regressão Logística",
        "Random Forest"
    ],

    "Accuracy": [
        accuracy_baseline,
        accuracy_logistico,
        accuracy_random_forest
    ],

    "Precision": [
        precision_baseline,
        precision_logistico,
        precision_random_forest
    ],

    "Recall": [
        recall_baseline,
        recall_logistico,
        recall_random_forest
    ],

    "F1-score": [
        f1_baseline,
        f1_logistico,
        f1_random_forest
    ],

    "ROC AUC": [
        roc_auc_baseline,
        roc_auc_logistico,
        roc_auc_random_forest
    ]
})

comparacao_modelos.round(4)

## Visualização das métricas

O gráfico abaixo compara precision, recall, F1-score e ROC AUC.

Como as métricas possuem interpretações diferentes, o melhor modelo depende do objetivo do negócio.

Para retenção de clientes, recall costuma ter importância elevada.

In [ ]:
metricas_comparacao = comparacao_modelos.set_index(
    "Modelo"
)[[
    "Precision",
    "Recall",
    "F1-score",
    "ROC AUC"
]]

metricas_comparacao.T.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title(
    "Comparação dos modelos de churn"
)

plt.xlabel("Métrica")
plt.ylabel("Resultado")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Modelo")
plt.show()

# 28. Comparação das curvas ROC

A comparação das curvas ROC permite visualizar qual modelo apresenta melhor separação entre as classes em diferentes thresholds.

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 6)
)

RocCurveDisplay.from_predictions(
    y_test,
    y_proba_logistico,
    name="Regressão Logística",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_proba_random_forest,
    name="Random Forest",
    ax=ax
)

plt.title(
    "Comparação das Curvas ROC"
)

plt.show()

# 29. Importância das variáveis no Random Forest

O Random Forest atribui uma importância a cada variável transformada.

Essa medida indica quanto cada feature contribuiu para as divisões das árvores.

Ela não representa causalidade.

Variáveis categóricas transformadas com One-Hot Encoding aparecem como várias colunas separadas.

In [ ]:
nomes_features_rf = modelo_random_forest.named_steps[
    "preprocessador"
].get_feature_names_out()

importancias_rf = modelo_random_forest.named_steps[
    "modelo"
].feature_importances_

df_importancias_rf = pd.DataFrame({
    "Feature": nomes_features_rf,
    "Importancia": importancias_rf
})

df_importancias_rf = df_importancias_rf.sort_values(
    by="Importancia",
    ascending=False
)

df_importancias_rf.head(15)

## Visualização das 15 variáveis mais importantes

In [ ]:
top_importancias = df_importancias_rf.head(15)

plt.figure(
    figsize=(10, 7)
)

plt.barh(
    top_importancias["Feature"][::-1],
    top_importancias["Importancia"][::-1]
)

plt.xlabel("Importância")
plt.ylabel("Feature")
plt.title(
    "15 variáveis mais importantes — Random Forest"
)

plt.show()

# 30. Verificação de possível overfitting

Vamos comparar o desempenho do Random Forest nos conjuntos de treino e teste.

Uma diferença muito grande pode indicar overfitting.

In [ ]:
y_pred_rf_treino = modelo_random_forest.predict(
    X_train
)

accuracy_rf_treino = accuracy_score(
    y_train,
    y_pred_rf_treino
)

f1_rf_treino = f1_score(
    y_train,
    y_pred_rf_treino,
    zero_division=0
)

print("===== RANDOM FOREST: TREINO VS TESTE =====")
print(f"Accuracy treino: {accuracy_rf_treino:.4f}")
print(f"Accuracy teste:  {accuracy_random_forest:.4f}")
print(f"F1 treino:       {f1_rf_treino:.4f}")
print(f"F1 teste:        {f1_random_forest:.4f}")

## Interpretação

Se o desempenho de treino for muito superior ao desempenho de teste, o modelo pode estar memorizando os dados.

Nesse caso, podemos testar:

- limite de profundidade;
- aumento de `min_samples_leaf`;
- aumento de `min_samples_split`;
- redução da complexidade;
- validação cruzada.

# 31. Random Forest regularizado

Vamos criar uma segunda versão mais controlada para reduzir o risco de overfitting.

Alterações:

- `max_depth=8`;
- `min_samples_split=10`;
- `min_samples_leaf=5`;
- `class_weight="balanced"`.

In [ ]:
modelo_random_forest_regularizado = Pipeline([
    (
        "preprocessador",
        preprocessador
    ),
    (
        "modelo",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_split=10,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )
    )
])

modelo_random_forest_regularizado.fit(
    X_train,
    y_train
)

print("Random Forest regularizado treinado com sucesso.")

In [ ]:
y_pred_rf_regularizado = (
    modelo_random_forest_regularizado.predict(
        X_test
    )
)

y_proba_rf_regularizado = (
    modelo_random_forest_regularizado.predict_proba(
        X_test
    )[:, 1]
)

accuracy_rf_regularizado = accuracy_score(
    y_test,
    y_pred_rf_regularizado
)

precision_rf_regularizado = precision_score(
    y_test,
    y_pred_rf_regularizado,
    zero_division=0
)

recall_rf_regularizado = recall_score(
    y_test,
    y_pred_rf_regularizado,
    zero_division=0
)

f1_rf_regularizado = f1_score(
    y_test,
    y_pred_rf_regularizado,
    zero_division=0
)

roc_auc_rf_regularizado = roc_auc_score(
    y_test,
    y_proba_rf_regularizado
)

print("===== RANDOM FOREST REGULARIZADO =====")
print(f"Accuracy:  {accuracy_rf_regularizado:.4f}")
print(f"Precision: {precision_rf_regularizado:.4f}")
print(f"Recall:    {recall_rf_regularizado:.4f}")
print(f"F1-score:  {f1_rf_regularizado:.4f}")
print(f"ROC AUC:   {roc_auc_rf_regularizado:.4f}")

# 32. Comparação final desta etapa

Agora incluiremos o Random Forest regularizado na comparação.

In [ ]:
comparacao_modelos = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regressão Logística",
        "Random Forest",
        "Random Forest regularizado"
    ],

    "Accuracy": [
        accuracy_baseline,
        accuracy_logistico,
        accuracy_random_forest,
        accuracy_rf_regularizado
    ],

    "Precision": [
        precision_baseline,
        precision_logistico,
        precision_random_forest,
        precision_rf_regularizado
    ],

    "Recall": [
        recall_baseline,
        recall_logistico,
        recall_random_forest,
        recall_rf_regularizado
    ],

    "F1-score": [
        f1_baseline,
        f1_logistico,
        f1_random_forest,
        f1_rf_regularizado
    ],

    "ROC AUC": [
        roc_auc_baseline,
        roc_auc_logistico,
        roc_auc_random_forest,
        roc_auc_rf_regularizado
    ]
})

comparacao_modelos.sort_values(
    by="F1-score",
    ascending=False
).round(4)

# 34. Rede neural para classificação binária

Agora construiremos uma rede neural para prever churn.

Diferentemente do projeto anterior, que era de regressão, neste projeto queremos prever uma das duas classes:

```text
0 → cliente permanece
1 → cliente cancela
```

Por isso, a camada de saída utilizará a função de ativação **Sigmoid**.

A Sigmoid transforma a saída da rede em um valor entre 0 e 1, que pode ser interpretado como probabilidade de churn.

## Preparação dos dados para a rede neural

Os modelos anteriores utilizaram um `Pipeline`, que aplicava o pré-processamento internamente.

Para a rede neural, vamos aplicar o `preprocessador` separadamente e gerar matrizes numéricas prontas para o TensorFlow.

O pré-processador será ajustado somente com o conjunto de treinamento.

Depois, ele será aplicado aos conjuntos de validação e teste.

In [ ]:
from sklearn.base import clone

preprocessador_rede = clone(
    preprocessador
)

X_train_processado = preprocessador_rede.fit_transform(
    X_train
)

X_val_processado = preprocessador_rede.transform(
    X_val
)

X_test_processado = preprocessador_rede.transform(
    X_test
)

print("Shape de treino:", X_train_processado.shape)
print("Shape de validação:", X_val_processado.shape)
print("Shape de teste:", X_test_processado.shape)

## Conversão dos alvos

O TensorFlow consegue trabalhar com Series do Pandas, mas vamos convertê-las para arrays NumPy para deixar o formato explícito.

In [ ]:
y_train_array = y_train.to_numpy()
y_val_array = y_val.to_numpy()
y_test_array = y_test.to_numpy()

print("Shape de y_train:", y_train_array.shape)
print("Shape de y_val:", y_val_array.shape)
print("Shape de y_test:", y_test_array.shape)

# 35. Arquitetura da rede neural

A arquitetura inicial será:

```text
Entradas processadas
        ↓
64 neurônios — ReLU
        ↓
Dropout 20%
        ↓
32 neurônios — ReLU
        ↓
Dropout 20%
        ↓
16 neurônios — ReLU
        ↓
1 neurônio — Sigmoid
```

A função Sigmoid será usada na camada de saída porque estamos resolvendo um problema de classificação binária.

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

modelo_rede_neural = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(X_train_processado.shape[1],)
    ),

    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(
        16,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

modelo_rede_neural.summary()

# 36. Compilação da rede neural

Utilizaremos:

### Adam

Otimizador responsável por atualizar os pesos da rede.

### Binary Crossentropy

Função de perda adequada para classificação binária.

### Métricas

Durante o treinamento, acompanharemos:

- accuracy;
- precision;
- recall;
- ROC AUC.

In [ ]:
modelo_rede_neural.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="binary_crossentropy",

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="recall"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

# 37. Early Stopping

O Early Stopping interromperá o treinamento quando o `val_loss` deixar de melhorar por 20 épocas consecutivas.

Ao final, os melhores pesos serão restaurados.

In [ ]:
early_stopping_rede = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

# 38. Treinamento da rede neural

Usaremos:

- até 300 épocas;
- batch size de 32;
- conjunto de validação separado;
- Early Stopping.

In [ ]:
history_rede = modelo_rede_neural.fit(
    X_train_processado,
    y_train_array,

    validation_data=(
        X_val_processado,
        y_val_array
    ),

    epochs=300,
    batch_size=32,

    callbacks=[
        early_stopping_rede
    ],

    verbose=1
)

# 39. Curvas de treinamento

As curvas de loss ajudam a observar:

- convergência;
- possível overfitting;
- possível underfitting;
- momento em que o Early Stopping interrompeu o treinamento.

In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.plot(
    history_rede.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_rede.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Épocas")
plt.ylabel("Binary Crossentropy")
plt.title("Loss da rede neural")
plt.legend()
plt.show()

## Curvas de AUC

A AUC mostra a capacidade de separação entre as classes ao longo do treinamento.

In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.plot(
    history_rede.history["auc"],
    label="Training AUC"
)

plt.plot(
    history_rede.history["val_auc"],
    label="Validation AUC"
)

plt.xlabel("Épocas")
plt.ylabel("AUC")
plt.title("AUC da rede neural")
plt.legend()
plt.show()

# 40. Probabilidades e classes previstas

A rede neural retorna probabilidades.

Usaremos inicialmente o threshold padrão de 0,5:

```text
probabilidade < 0,5 → permanece
probabilidade ≥ 0,5 → churn
```

In [ ]:
y_proba_rede = modelo_rede_neural.predict(
    X_test_processado,
    verbose=0
).flatten()

threshold_padrao = 0.5

y_pred_rede = (
    y_proba_rede >= threshold_padrao
).astype(int)

print("Primeiras probabilidades:")
print(
    np.round(
        y_proba_rede[:10],
        4
    )
)

print("\nPrimeiras classes previstas:")
print(y_pred_rede[:10])

# 41. Avaliação da rede neural

Vamos calcular as mesmas métricas utilizadas nos outros modelos.

In [ ]:
accuracy_rede = accuracy_score(
    y_test,
    y_pred_rede
)

precision_rede = precision_score(
    y_test,
    y_pred_rede,
    zero_division=0
)

recall_rede = recall_score(
    y_test,
    y_pred_rede,
    zero_division=0
)

f1_rede = f1_score(
    y_test,
    y_pred_rede,
    zero_division=0
)

roc_auc_rede = roc_auc_score(
    y_test,
    y_proba_rede
)

print("===== REDE NEURAL =====")
print(f"Accuracy:  {accuracy_rede:.4f}")
print(f"Precision: {precision_rede:.4f}")
print(f"Recall:    {recall_rede:.4f}")
print(f"F1-score:  {f1_rede:.4f}")
print(f"ROC AUC:   {roc_auc_rede:.4f}")
print(
    "Épocas executadas:",
    len(history_rede.history["loss"])
)

## Relatório de classificação da rede neural

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_rede,
        target_names=[
            "Permanece",
            "Churn"
        ],
        zero_division=0
    )
)

# 42. Matriz de confusão da rede neural

A matriz de confusão permitirá verificar quantos clientes em churn foram identificados e quantos não foram detectados.

In [ ]:
matriz_rede = confusion_matrix(
    y_test,
    y_pred_rede
)

display_matriz_rede = ConfusionMatrixDisplay(
    confusion_matrix=matriz_rede,
    display_labels=[
        "Permanece",
        "Churn"
    ]
)

display_matriz_rede.plot(
    values_format="d"
)

plt.title(
    "Matriz de Confusão — Rede Neural"
)

plt.show()

In [ ]:
tn_rede, fp_rede, fn_rede, tp_rede = matriz_rede.ravel()

print("Verdadeiros negativos:", tn_rede)
print("Falsos positivos:", fp_rede)
print("Falsos negativos:", fn_rede)
print("Verdadeiros positivos:", tp_rede)

# 43. Curva ROC da rede neural

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_proba_rede,
    name="Rede Neural"
)

plt.title(
    "Curva ROC — Rede Neural"
)

plt.show()

# 44. Comparação de todos os modelos

Agora vamos comparar:

1. baseline;
2. Regressão Logística;
3. Random Forest;
4. Random Forest regularizado;
5. rede neural.

In [ ]:
comparacao_modelos = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regressão Logística",
        "Random Forest",
        "Random Forest regularizado",
        "Rede Neural"
    ],

    "Accuracy": [
        accuracy_baseline,
        accuracy_logistico,
        accuracy_random_forest,
        accuracy_rf_regularizado,
        accuracy_rede
    ],

    "Precision": [
        precision_baseline,
        precision_logistico,
        precision_random_forest,
        precision_rf_regularizado,
        precision_rede
    ],

    "Recall": [
        recall_baseline,
        recall_logistico,
        recall_random_forest,
        recall_rf_regularizado,
        recall_rede
    ],

    "F1-score": [
        f1_baseline,
        f1_logistico,
        f1_random_forest,
        f1_rf_regularizado,
        f1_rede
    ],

    "ROC AUC": [
        roc_auc_baseline,
        roc_auc_logistico,
        roc_auc_random_forest,
        roc_auc_rf_regularizado,
        roc_auc_rede
    ]
})

comparacao_modelos.sort_values(
    by="F1-score",
    ascending=False
).round(4)

## Visualização das métricas finais

O gráfico compara precision, recall, F1-score e ROC AUC.

Nenhuma métrica deve ser analisada isoladamente.

In [ ]:
metricas_finais = comparacao_modelos.set_index(
    "Modelo"
)[[
    "Precision",
    "Recall",
    "F1-score",
    "ROC AUC"
]]

metricas_finais.T.plot(
    kind="bar",
    figsize=(12, 7)
)

plt.title(
    "Comparação final dos modelos"
)

plt.xlabel("Métrica")
plt.ylabel("Resultado")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Modelo")
plt.show()

# 45. Comparação final das curvas ROC

Vamos comparar as curvas ROC dos três modelos reais principais.

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 7)
)

RocCurveDisplay.from_predictions(
    y_test,
    y_proba_logistico,
    name="Regressão Logística",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_proba_random_forest,
    name="Random Forest",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_proba_rf_regularizado,
    name="Random Forest regularizado",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_proba_rede,
    name="Rede Neural",
    ax=ax
)

plt.title(
    "Curvas ROC dos modelos"
)

plt.show()

# 46. Seleção preliminar do melhor modelo

Nesta etapa, vamos identificar automaticamente:

- melhor F1-score;
- melhor recall;
- melhor ROC AUC.

O modelo ideal pode mudar conforme o objetivo do negócio.

In [ ]:
melhor_f1 = comparacao_modelos.loc[
    comparacao_modelos["F1-score"].idxmax()
]

melhor_recall = comparacao_modelos.loc[
    comparacao_modelos["Recall"].idxmax()
]

melhor_auc = comparacao_modelos.loc[
    comparacao_modelos["ROC AUC"].idxmax()
]

print("===== MELHORES RESULTADOS =====")

print(
    f"Melhor F1-score: "
    f"{melhor_f1['Modelo']} "
    f"({melhor_f1['F1-score']:.4f})"
)

print(
    f"Melhor Recall: "
    f"{melhor_recall['Modelo']} "
    f"({melhor_recall['Recall']:.4f})"
)

print(
    f"Melhor ROC AUC: "
    f"{melhor_auc['Modelo']} "
    f"({melhor_auc['ROC AUC']:.4f})"
)

# 47. Ajuste do threshold de classificação

Até agora, usamos o threshold padrão de `0,5`.

Isso significa:

```text
probabilidade < 0,5 → permanece
probabilidade ≥ 0,5 → churn
```

Entretanto, em problemas de retenção, o melhor threshold pode ser diferente.

Reduzir o threshold costuma:

- aumentar o recall;
- identificar mais clientes em risco;
- reduzir falsos negativos;
- aumentar falsos positivos;
- reduzir precision.

A escolha do threshold depende do custo de cada erro para o negócio.

## Função para avaliar diferentes thresholds

A função abaixo calcula, para cada threshold:

- accuracy;
- precision;
- recall;
- F1-score;
- falsos positivos;
- falsos negativos;
- verdadeiros positivos;
- verdadeiros negativos.

In [ ]:
def avaliar_thresholds(
    y_real,
    probabilidades,
    thresholds
):
    resultados = []

    for threshold in thresholds:
        previsoes = (
            probabilidades >= threshold
        ).astype(int)

        matriz = confusion_matrix(
            y_real,
            previsoes
        )

        tn, fp, fn, tp = matriz.ravel()

        resultados.append({
            "Threshold": threshold,
            "Accuracy": accuracy_score(
                y_real,
                previsoes
            ),
            "Precision": precision_score(
                y_real,
                previsoes,
                zero_division=0
            ),
            "Recall": recall_score(
                y_real,
                previsoes,
                zero_division=0
            ),
            "F1-score": f1_score(
                y_real,
                previsoes,
                zero_division=0
            ),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp
        })

    return pd.DataFrame(resultados)

# 48. Teste de thresholds na validação

O threshold deve ser escolhido com base no conjunto de validação, e não no conjunto de teste.

Primeiro, vamos gerar probabilidades para o conjunto de validação.

In [ ]:
y_proba_rede_val = modelo_rede_neural.predict(
    X_val_processado,
    verbose=0
).flatten()

thresholds = np.arange(
    0.10,
    0.91,
    0.05
)

resultados_thresholds_val = avaliar_thresholds(
    y_val,
    y_proba_rede_val,
    thresholds
)

resultados_thresholds_val.round(4)

## Visualização das métricas por threshold

O gráfico abaixo mostra como precision, recall e F1-score mudam conforme o threshold.

In [ ]:
plt.figure(
    figsize=(11, 6)
)

plt.plot(
    resultados_thresholds_val["Threshold"],
    resultados_thresholds_val["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    resultados_thresholds_val["Threshold"],
    resultados_thresholds_val["Recall"],
    marker="o",
    label="Recall"
)

plt.plot(
    resultados_thresholds_val["Threshold"],
    resultados_thresholds_val["F1-score"],
    marker="o",
    label="F1-score"
)

plt.xlabel("Threshold")
plt.ylabel("Resultado")
plt.title(
    "Precision, Recall e F1 por threshold"
)
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.show()

# 49. Escolha automática do threshold pelo melhor F1-score

Uma forma objetiva de escolher o threshold é maximizar o F1-score no conjunto de validação.

Essa escolha busca equilíbrio entre precision e recall.

In [ ]:
indice_melhor_f1 = (
    resultados_thresholds_val["F1-score"].idxmax()
)

melhor_threshold_f1 = resultados_thresholds_val.loc[
    indice_melhor_f1,
    "Threshold"
]

melhor_resultado_f1_val = resultados_thresholds_val.loc[
    indice_melhor_f1
]

print("===== MELHOR THRESHOLD POR F1 =====")
print(
    f"Threshold: "
    f"{melhor_threshold_f1:.2f}"
)
print(
    f"Precision: "
    f"{melhor_resultado_f1_val['Precision']:.4f}"
)
print(
    f"Recall: "
    f"{melhor_resultado_f1_val['Recall']:.4f}"
)
print(
    f"F1-score: "
    f"{melhor_resultado_f1_val['F1-score']:.4f}"
)
print(
    f"Falsos negativos: "
    f"{int(melhor_resultado_f1_val['FN'])}"
)

# 50. Threshold orientado a recall

Em algumas empresas, perder um cliente em risco pode ser mais caro do que abordar um cliente que não cancelaria.

Nesse caso, podemos escolher o menor threshold que atinja um recall mínimo.

Neste exemplo, usaremos recall mínimo de `0,80`.

Esse valor é didático e deve ser definido com base no contexto de negócio.

In [ ]:
recall_minimo = 0.80

candidatos_recall = resultados_thresholds_val[
    resultados_thresholds_val["Recall"] >= recall_minimo
].copy()

if not candidatos_recall.empty:
    melhor_recall_negocio = candidatos_recall.sort_values(
        by=[
            "Precision",
            "F1-score"
        ],
        ascending=False
    ).iloc[0]

    threshold_recall_negocio = (
        melhor_recall_negocio["Threshold"]
    )

    print(
        "===== THRESHOLD ORIENTADO A RECALL ====="
    )
    print(
        f"Threshold: "
        f"{threshold_recall_negocio:.2f}"
    )
    print(
        f"Precision: "
        f"{melhor_recall_negocio['Precision']:.4f}"
    )
    print(
        f"Recall: "
        f"{melhor_recall_negocio['Recall']:.4f}"
    )
    print(
        f"F1-score: "
        f"{melhor_recall_negocio['F1-score']:.4f}"
    )
    print(
        f"Falsos negativos: "
        f"{int(melhor_recall_negocio['FN'])}"
    )
else:
    threshold_recall_negocio = melhor_threshold_f1

    print(
        "Nenhum threshold atingiu o recall mínimo."
    )
    print(
        "Será usado o melhor threshold por F1-score."
    )

# 51. Avaliação final do threshold escolhido no conjunto de teste

Depois de escolher o threshold com o conjunto de validação, aplicamos esse valor uma única vez ao conjunto de teste.

Para a avaliação principal, usaremos o threshold que maximiza o F1-score.

In [ ]:
threshold_final = float(
    melhor_threshold_f1
)

y_pred_rede_threshold = (
    y_proba_rede >= threshold_final
).astype(int)

accuracy_rede_threshold = accuracy_score(
    y_test,
    y_pred_rede_threshold
)

precision_rede_threshold = precision_score(
    y_test,
    y_pred_rede_threshold,
    zero_division=0
)

recall_rede_threshold = recall_score(
    y_test,
    y_pred_rede_threshold,
    zero_division=0
)

f1_rede_threshold = f1_score(
    y_test,
    y_pred_rede_threshold,
    zero_division=0
)

roc_auc_rede_threshold = roc_auc_score(
    y_test,
    y_proba_rede
)

print(
    "===== REDE NEURAL COM THRESHOLD AJUSTADO ====="
)
print(f"Threshold: {threshold_final:.2f}")
print(f"Accuracy:  {accuracy_rede_threshold:.4f}")
print(f"Precision: {precision_rede_threshold:.4f}")
print(f"Recall:    {recall_rede_threshold:.4f}")
print(f"F1-score:  {f1_rede_threshold:.4f}")
print(f"ROC AUC:   {roc_auc_rede_threshold:.4f}")

## Comparação do threshold padrão com o ajustado

A ROC AUC permanece igual, pois ela é calculada com as probabilidades e não depende de um único threshold.

In [ ]:
comparacao_thresholds = pd.DataFrame({
    "Configuração": [
        "Threshold 0,50",
        f"Threshold {threshold_final:.2f}"
    ],

    "Accuracy": [
        accuracy_rede,
        accuracy_rede_threshold
    ],

    "Precision": [
        precision_rede,
        precision_rede_threshold
    ],

    "Recall": [
        recall_rede,
        recall_rede_threshold
    ],

    "F1-score": [
        f1_rede,
        f1_rede_threshold
    ],

    "ROC AUC": [
        roc_auc_rede,
        roc_auc_rede_threshold
    ]
})

comparacao_thresholds.round(4)

# 52. Matriz de confusão com threshold ajustado

Agora podemos observar diretamente o efeito do novo threshold sobre falsos positivos e falsos negativos.

In [ ]:
matriz_rede_threshold = confusion_matrix(
    y_test,
    y_pred_rede_threshold
)

display_threshold = ConfusionMatrixDisplay(
    confusion_matrix=matriz_rede_threshold,
    display_labels=[
        "Permanece",
        "Churn"
    ]
)

display_threshold.plot(
    values_format="d"
)

plt.title(
    "Matriz de Confusão — Threshold ajustado"
)

plt.show()

In [ ]:
tn_t, fp_t, fn_t, tp_t = (
    matriz_rede_threshold.ravel()
)

print("Verdadeiros negativos:", tn_t)
print("Falsos positivos:", fp_t)
print("Falsos negativos:", fn_t)
print("Verdadeiros positivos:", tp_t)

# 53. Comparação das matrizes de confusão

A tabela abaixo mostra o efeito do ajuste do threshold sobre os tipos de erro.

In [ ]:
comparacao_erros_threshold = pd.DataFrame({
    "Configuração": [
        "Threshold 0,50",
        f"Threshold {threshold_final:.2f}"
    ],

    "Verdadeiros negativos": [
        tn_rede,
        tn_t
    ],

    "Falsos positivos": [
        fp_rede,
        fp_t
    ],

    "Falsos negativos": [
        fn_rede,
        fn_t
    ],

    "Verdadeiros positivos": [
        tp_rede,
        tp_t
    ]
})

comparacao_erros_threshold

# 54. Atualização da comparação final dos modelos

Vamos incluir a rede neural com threshold ajustado na tabela geral.

Isso permite comparar o efeito do threshold com os outros modelos.

In [ ]:
comparacao_modelos_threshold = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regressão Logística",
        "Random Forest",
        "Random Forest regularizado",
        "Rede Neural — threshold 0,50",
        (
            "Rede Neural — threshold "
            f"{threshold_final:.2f}"
        )
    ],

    "Accuracy": [
        accuracy_baseline,
        accuracy_logistico,
        accuracy_random_forest,
        accuracy_rf_regularizado,
        accuracy_rede,
        accuracy_rede_threshold
    ],

    "Precision": [
        precision_baseline,
        precision_logistico,
        precision_random_forest,
        precision_rf_regularizado,
        precision_rede,
        precision_rede_threshold
    ],

    "Recall": [
        recall_baseline,
        recall_logistico,
        recall_random_forest,
        recall_rf_regularizado,
        recall_rede,
        recall_rede_threshold
    ],

    "F1-score": [
        f1_baseline,
        f1_logistico,
        f1_random_forest,
        f1_rf_regularizado,
        f1_rede,
        f1_rede_threshold
    ],

    "ROC AUC": [
        roc_auc_baseline,
        roc_auc_logistico,
        roc_auc_random_forest,
        roc_auc_rf_regularizado,
        roc_auc_rede,
        roc_auc_rede_threshold
    ]
})

comparacao_modelos_threshold.sort_values(
    by="F1-score",
    ascending=False
).round(4)

# 55. Seleção do melhor modelo conforme o objetivo

Não existe um único modelo melhor para todos os cenários.

Vamos identificar:

- melhor F1-score;
- melhor recall;
- melhor precision;
- melhor ROC AUC.

In [ ]:
melhor_f1_final = comparacao_modelos_threshold.loc[
    comparacao_modelos_threshold[
        "F1-score"
    ].idxmax()
]

melhor_recall_final = comparacao_modelos_threshold.loc[
    comparacao_modelos_threshold[
        "Recall"
    ].idxmax()
]

melhor_precision_final = comparacao_modelos_threshold.loc[
    comparacao_modelos_threshold[
        "Precision"
    ].idxmax()
]

melhor_auc_final = comparacao_modelos_threshold.loc[
    comparacao_modelos_threshold[
        "ROC AUC"
    ].idxmax()
]

print("===== MELHORES MODELOS =====")

print(
    f"Melhor F1-score: "
    f"{melhor_f1_final['Modelo']} "
    f"({melhor_f1_final['F1-score']:.4f})"
)

print(
    f"Melhor Recall: "
    f"{melhor_recall_final['Modelo']} "
    f"({melhor_recall_final['Recall']:.4f})"
)

print(
    f"Melhor Precision: "
    f"{melhor_precision_final['Modelo']} "
    f"({melhor_precision_final['Precision']:.4f})"
)

print(
    f"Melhor ROC AUC: "
    f"{melhor_auc_final['Modelo']} "
    f"({melhor_auc_final['ROC AUC']:.4f})"
)

# 56. Definição do modelo final

A escolha do modelo final deve refletir o objetivo do negócio.

Neste projeto, utilizaremos como critério principal o **melhor F1-score**, pois ele equilibra precision e recall.

Isso evita escolher um modelo que:

- identifique muitos clientes em risco, mas gere excesso de falsos positivos;
- tenha alta precision, mas deixe muitos clientes em churn sem identificação.

O modelo final será definido automaticamente a partir da tabela `comparacao_modelos_threshold`.

In [ ]:
linha_modelo_final = comparacao_modelos_threshold.loc[
    comparacao_modelos_threshold["F1-score"].idxmax()
]

nome_modelo_final = linha_modelo_final["Modelo"]

print("===== MODELO FINAL SELECIONADO =====")
print(f"Modelo: {nome_modelo_final}")
print(f"Accuracy:  {linha_modelo_final['Accuracy']:.4f}")
print(f"Precision: {linha_modelo_final['Precision']:.4f}")
print(f"Recall:    {linha_modelo_final['Recall']:.4f}")
print(f"F1-score:  {linha_modelo_final['F1-score']:.4f}")
print(f"ROC AUC:   {linha_modelo_final['ROC AUC']:.4f}")

## Mapeamento entre nomes e objetos treinados

Agora relacionaremos o nome do modelo selecionado ao objeto correspondente.

Para modelos baseados em pipeline, o pré-processamento já está incorporado.

Para a rede neural, o modelo e o pré-processador são salvos separadamente.

In [ ]:
modelos_disponiveis = {
    "Baseline": modelo_baseline,
    "Regressão Logística": modelo_logistico,
    "Random Forest": modelo_random_forest,
    "Random Forest regularizado": modelo_random_forest_regularizado
}

modelo_final_tipo = None
modelo_final = None
preprocessador_final = None
threshold_modelo_final = 0.5

if nome_modelo_final in modelos_disponiveis:
    modelo_final_tipo = "pipeline_sklearn"
    modelo_final = modelos_disponiveis[
        nome_modelo_final
    ]

elif nome_modelo_final.startswith(
    "Rede Neural"
):
    modelo_final_tipo = "rede_neural"
    modelo_final = modelo_rede_neural
    preprocessador_final = preprocessador_rede

    if "threshold 0,50" in nome_modelo_final:
        threshold_modelo_final = 0.5
    else:
        threshold_modelo_final = threshold_final

else:
    raise ValueError(
        "O modelo final selecionado não foi reconhecido."
    )

print(
    f"Tipo do artefato final: "
    f"{modelo_final_tipo}"
)

print(
    f"Threshold final: "
    f"{threshold_modelo_final:.2f}"
)

# 57. Criação da pasta de artefatos

Vamos salvar todos os arquivos necessários em:

```text
artefatos_projeto_02/
```

Os artefatos incluirão:

- modelo final;
- pré-processador, quando necessário;
- threshold;
- nomes das colunas;
- tabela final de métricas;
- metadados do projeto.

In [ ]:
from pathlib import Path
import json
import joblib

pasta_artefatos = Path(
    "artefatos_projeto_02"
)

pasta_artefatos.mkdir(
    exist_ok=True
)

print(
    f"Pasta criada: "
    f"{pasta_artefatos.resolve()}"
)

# 58. Salvamento do modelo final

## Modelos Scikit-learn

Os pipelines serão salvos com `joblib`.

Como o pipeline contém pré-processamento e modelo, basta carregar um único arquivo para realizar a inferência.

## Rede neural

Caso a rede neural seja selecionada:

- o modelo será salvo no formato `.keras`;
- o pré-processador será salvo com `joblib`;
- o threshold será salvo separadamente.

In [ ]:
caminho_modelo_sklearn = (
    pasta_artefatos
    / "modelo_final_churn.pkl"
)

caminho_modelo_keras = (
    pasta_artefatos
    / "modelo_final_churn.keras"
)

caminho_preprocessador = (
    pasta_artefatos
    / "preprocessador_churn.pkl"
)

if modelo_final_tipo == "pipeline_sklearn":
    joblib.dump(
        modelo_final,
        caminho_modelo_sklearn
    )

    print(
        f"Pipeline salvo em: "
        f"{caminho_modelo_sklearn}"
    )

elif modelo_final_tipo == "rede_neural":
    modelo_final.save(
        caminho_modelo_keras
    )

    joblib.dump(
        preprocessador_final,
        caminho_preprocessador
    )

    print(
        f"Rede neural salva em: "
        f"{caminho_modelo_keras}"
    )

    print(
        f"Pré-processador salvo em: "
        f"{caminho_preprocessador}"
    )

# 59. Salvamento do threshold

O threshold será armazenado em um arquivo JSON.

Isso permite que a aplicação utilize exatamente o mesmo limiar escolhido durante a validação.

In [ ]:
caminho_threshold = (
    pasta_artefatos
    / "threshold.json"
)

dados_threshold = {
    "threshold": float(
        threshold_modelo_final
    ),
    "modelo": nome_modelo_final
}

with caminho_threshold.open(
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        dados_threshold,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Threshold salvo em: "
    f"{caminho_threshold}"
)

# 60. Salvamento das colunas de entrada

Uma aplicação precisa receber as mesmas colunas utilizadas durante o treinamento.

Vamos salvar:

- nomes das colunas;
- tipos básicos;
- ordem original.

Isso reduz o risco de erro durante a inferência.

In [ ]:
caminho_colunas = (
    pasta_artefatos
    / "colunas_entrada.json"
)

dados_colunas = {
    "colunas": X.columns.tolist(),
    "colunas_numericas": colunas_numericas,
    "colunas_categoricas": colunas_categoricas
}

with caminho_colunas.open(
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        dados_colunas,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Colunas salvas em: "
    f"{caminho_colunas}"
)

# 61. Salvamento da tabela final de resultados

A tabela de comparação será salva em CSV.

Ela poderá ser exibida na interface ou utilizada na documentação do projeto.

In [ ]:
caminho_resultados = (
    pasta_artefatos
    / "comparacao_modelos.csv"
)

comparacao_modelos_threshold.to_csv(
    caminho_resultados,
    index=False
)

print(
    f"Resultados salvos em: "
    f"{caminho_resultados}"
)

# 62. Salvamento dos metadados do projeto

Os metadados registram:

- nome do modelo final;
- tipo do modelo;
- threshold;
- métricas principais;
- quantidade de features;
- seed utilizada.

Essas informações facilitam auditoria e reprodução do projeto.

In [ ]:
caminho_metadados = (
    pasta_artefatos
    / "metadados_modelo.json"
)

metadados = {
    "projeto": (
        "Projeto 02 — Previsão de Churn"
    ),
    "modelo_final": nome_modelo_final,
    "tipo_modelo": modelo_final_tipo,
    "threshold": float(
        threshold_modelo_final
    ),
    "seed": int(SEED),
    "numero_features_originais": int(
        X.shape[1]
    ),
    "metricas": {
        "accuracy": float(
            linha_modelo_final["Accuracy"]
        ),
        "precision": float(
            linha_modelo_final["Precision"]
        ),
        "recall": float(
            linha_modelo_final["Recall"]
        ),
        "f1_score": float(
            linha_modelo_final["F1-score"]
        ),
        "roc_auc": float(
            linha_modelo_final["ROC AUC"]
        )
    }
}

with caminho_metadados.open(
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        metadados,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Metadados salvos em: "
    f"{caminho_metadados}"
)

# 63. Teste de carregamento dos artefatos

Salvar os arquivos não é suficiente.

Também precisamos verificar se eles podem ser carregados e utilizados novamente.

Vamos selecionar um cliente do conjunto de teste e simular uma previsão.

In [ ]:
exemplo_teste = X_test.iloc[[0]].copy()
valor_real_teste = int(
    y_test.iloc[0]
)

print("Exemplo selecionado:")
display(exemplo_teste)

print(
    f"Classe real: "
    f"{valor_real_teste}"
)

## Carregamento e inferência

A lógica muda conforme o tipo de modelo final.

In [ ]:
with caminho_threshold.open(
    "r",
    encoding="utf-8"
) as arquivo:
    threshold_carregado = json.load(
        arquivo
    )["threshold"]

if modelo_final_tipo == "pipeline_sklearn":
    pipeline_carregado = joblib.load(
        caminho_modelo_sklearn
    )

    probabilidade_carregada = (
        pipeline_carregado.predict_proba(
            exemplo_teste
        )[:, 1][0]
    )

elif modelo_final_tipo == "rede_neural":
    modelo_carregado = (
        tf.keras.models.load_model(
            caminho_modelo_keras
        )
    )

    preprocessador_carregado = joblib.load(
        caminho_preprocessador
    )

    exemplo_processado = (
        preprocessador_carregado.transform(
            exemplo_teste
        )
    )

    probabilidade_carregada = float(
        modelo_carregado.predict(
            exemplo_processado,
            verbose=0
        ).flatten()[0]
    )

classe_carregada = int(
    probabilidade_carregada
    >= threshold_carregado
)

print(
    f"Probabilidade de churn: "
    f"{probabilidade_carregada:.4f}"
)

print(
    f"Threshold utilizado: "
    f"{threshold_carregado:.2f}"
)

print(
    f"Classe prevista: "
    f"{classe_carregada}"
)

print(
    f"Classe real: "
    f"{valor_real_teste}"
)

# 64. Função de inferência reutilizável

A função abaixo centraliza a previsão.

Ela poderá ser adaptada para a interface Streamlit.

In [ ]:
def prever_churn(
    dados_cliente,
    tipo_modelo,
    threshold,
    pipeline=None,
    modelo_keras=None,
    preprocessador=None
):
    if tipo_modelo == "pipeline_sklearn":
        probabilidade = (
            pipeline.predict_proba(
                dados_cliente
            )[:, 1][0]
        )

    elif tipo_modelo == "rede_neural":
        dados_processados = (
            preprocessador.transform(
                dados_cliente
            )
        )

        probabilidade = float(
            modelo_keras.predict(
                dados_processados,
                verbose=0
            ).flatten()[0]
        )

    else:
        raise ValueError(
            "Tipo de modelo não reconhecido."
        )

    classe = int(
        probabilidade >= threshold
    )

    return {
        "probabilidade_churn": probabilidade,
        "classe_prevista": classe,
        "threshold": threshold
    }

In [ ]:
if modelo_final_tipo == "pipeline_sklearn":
    resultado_inferencia = prever_churn(
        dados_cliente=exemplo_teste,
        tipo_modelo=modelo_final_tipo,
        threshold=threshold_carregado,
        pipeline=pipeline_carregado
    )

else:
    resultado_inferencia = prever_churn(
        dados_cliente=exemplo_teste,
        tipo_modelo=modelo_final_tipo,
        threshold=threshold_carregado,
        modelo_keras=modelo_carregado,
        preprocessador=preprocessador_carregado
    )

resultado_inferencia

# 65. Preparação para a interface Streamlit

A estrutura recomendada será:

```text
projeto_02_churn/
│
├── app.py
├── artefatos_projeto_02/
│   ├── modelo_final_churn.pkl
│   ├── modelo_final_churn.keras
│   ├── preprocessador_churn.pkl
│   ├── threshold.json
│   ├── colunas_entrada.json
│   ├── comparacao_modelos.csv
│   └── metadados_modelo.json
│
├── requirements.txt
└── README.md
```

A pasta conterá apenas os arquivos correspondentes ao tipo de modelo escolhido.

## Funcionalidades recomendadas

A interface deverá:

- receber os dados do cliente;
- exibir a probabilidade de churn;
- classificar o risco como baixo, médio ou alto;
- mostrar o threshold utilizado;
- explicar que a previsão é probabilística;
- exibir um aviso de finalidade educacional;
- tratar erros de campos vazios ou inválidos.

## Faixas de risco sugeridas

Uma forma simples de apresentar o resultado:

```text
0% a 39%   → baixo risco
40% a 69%  → risco moderado
70% a 100% → alto risco
```

Essas faixas são apenas ilustrativas e não foram otimizadas neste projeto.

In [ ]:
def classificar_risco(
    probabilidade
):
    if probabilidade < 0.40:
        return "Baixo risco"

    if probabilidade < 0.70:
        return "Risco moderado"

    return "Alto risco"


print(
    classificar_risco(
        probabilidade_carregada
    )
)

# 66. Arquivo requirements.txt sugerido

Para executar a futura interface, utilize:

```text
numpy
pandas
scikit-learn
tensorflow
joblib
streamlit
matplotlib
```

As versões podem ser fixadas após confirmar quais versões foram utilizadas no ambiente de treinamento.

# 67. Compactação dos artefatos

No Google Colab, podemos compactar a pasta para facilitar o download.

In [ ]:
import shutil

caminho_zip = shutil.make_archive(
    "artefatos_projeto_02",
    "zip",
    pasta_artefatos
)

print(
    f"Arquivo ZIP criado: "
    f"{caminho_zip}"
)

# 68. Resumo automático do projeto

A célula abaixo gera um texto com o resultado final.

In [ ]:
print("RESUMO FINAL DO PROJETO 02")
print("-" * 60)

print(
    f"Modelo selecionado: "
    f"{nome_modelo_final}"
)

print(
    f"Threshold utilizado: "
    f"{threshold_modelo_final:.2f}"
)

print(
    f"Accuracy: "
    f"{linha_modelo_final['Accuracy']:.4f}"
)

print(
    f"Precision: "
    f"{linha_modelo_final['Precision']:.4f}"
)

print(
    f"Recall: "
    f"{linha_modelo_final['Recall']:.4f}"
)

print(
    f"F1-score: "
    f"{linha_modelo_final['F1-score']:.4f}"
)

print(
    f"ROC AUC: "
    f"{linha_modelo_final['ROC AUC']:.4f}"
)

# 69. Extensão avançada do Projeto 02

Nesta etapa, o projeto será ampliado com técnicas que aproximam o notebook de um fluxo profissional de modelagem:

1. validação cruzada estratificada;
2. Gradient Boosting;
3. XGBoost;
4. calibração de probabilidades;
5. explicabilidade com SHAP;
6. simulação financeira de campanhas de retenção.

Esses experimentos serão realizados antes da construção definitiva da interface Streamlit.

# 70. Instalação das bibliotecas adicionais

O Google Colab normalmente já possui o Scikit-learn, mas pode ser necessário instalar ou atualizar:

- `xgboost`;
- `shap`.

Execute a célula abaixo antes das novas importações.

In [ ]:
%pip install -q xgboost shap

# 71. Novas importações

Utilizaremos:

- `StratifiedKFold` e `cross_validate` para validação cruzada;
- `GradientBoostingClassifier`;
- `XGBClassifier`;
- métricas de calibração;
- SHAP para explicabilidade.

In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.ensemble import (
    GradientBoostingClassifier
)

from sklearn.calibration import (
    CalibratedClassifierCV,
    CalibrationDisplay,
    calibration_curve
)

from sklearn.metrics import (
    brier_score_loss,
    log_loss
)

from xgboost import XGBClassifier

import shap

print("Bibliotecas avançadas importadas com sucesso.")

# 72. Validação cruzada estratificada

Até agora, os modelos foram comparados principalmente em uma única divisão dos dados.

A validação cruzada divide o conjunto de desenvolvimento em diferentes partes e repete o treinamento várias vezes.

Utilizaremos `StratifiedKFold` com 5 folds para preservar a proporção de churn em cada divisão.

As métricas serão:

- accuracy;
- precision;
- recall;
- F1-score;
- ROC AUC.

O conjunto de teste continuará separado e não será usado na validação cruzada.

In [ ]:
cv_estratificado = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

metricas_cv = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

## Modelos avaliados na validação cruzada

Começaremos com os modelos tradicionais já criados:

- Regressão Logística;
- Random Forest;
- Random Forest regularizado.

O baseline não será incluído nesta etapa, pois já conhecemos sua função de referência.

In [ ]:
modelos_cv = {
    "Regressão Logística": modelo_logistico,
    "Random Forest": modelo_random_forest,
    "Random Forest regularizado": (
        modelo_random_forest_regularizado
    )
}

In [ ]:
resultados_cv_lista = []

for nome_modelo, modelo in modelos_cv.items():
    resultado = cross_validate(
        modelo,
        X_temp,
        y_temp,
        cv=cv_estratificado,
        scoring=metricas_cv,
        n_jobs=-1,
        return_train_score=False
    )

    resultados_cv_lista.append({
        "Modelo": nome_modelo,
        "Accuracy média": resultado[
            "test_accuracy"
        ].mean(),
        "Accuracy desvio": resultado[
            "test_accuracy"
        ].std(),
        "Precision média": resultado[
            "test_precision"
        ].mean(),
        "Precision desvio": resultado[
            "test_precision"
        ].std(),
        "Recall médio": resultado[
            "test_recall"
        ].mean(),
        "Recall desvio": resultado[
            "test_recall"
        ].std(),
        "F1 médio": resultado[
            "test_f1"
        ].mean(),
        "F1 desvio": resultado[
            "test_f1"
        ].std(),
        "ROC AUC médio": resultado[
            "test_roc_auc"
        ].mean(),
        "ROC AUC desvio": resultado[
            "test_roc_auc"
        ].std()
    })

resultados_cv = pd.DataFrame(
    resultados_cv_lista
)

resultados_cv.round(4)

## Interpretação da validação cruzada

Além da média, devemos observar o desvio padrão.

Um modelo com média ligeiramente maior, mas grande variação entre os folds, pode ser menos confiável do que outro com desempenho mais estável.

In [ ]:
resultados_cv_ordenados = resultados_cv.sort_values(
    by="F1 médio",
    ascending=False
).reset_index(drop=True)

resultados_cv_ordenados.round(4)

In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.bar(
    resultados_cv_ordenados["Modelo"],
    resultados_cv_ordenados["F1 médio"],
    yerr=resultados_cv_ordenados["F1 desvio"],
    capsize=5
)

plt.xlabel("Modelo")
plt.ylabel("F1 médio")
plt.title(
    "Validação cruzada — F1 médio e desvio padrão"
)
plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.show()

# 73. Gradient Boosting

O Gradient Boosting constrói árvores sequencialmente.

Cada nova árvore procura corrigir parte dos erros cometidos pelas árvores anteriores.

Isso permite capturar relações não lineares e interações complexas entre as variáveis.

In [ ]:
modelo_gradient_boosting = Pipeline([
    (
        "preprocessador",
        preprocessador
    ),
    (
        "modelo",
        GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=SEED
        )
    )
])

modelo_gradient_boosting.fit(
    X_train,
    y_train
)

print("Gradient Boosting treinado com sucesso.")

In [ ]:
y_pred_gradient = modelo_gradient_boosting.predict(
    X_test
)

y_proba_gradient = (
    modelo_gradient_boosting.predict_proba(
        X_test
    )[:, 1]
)

accuracy_gradient = accuracy_score(
    y_test,
    y_pred_gradient
)

precision_gradient = precision_score(
    y_test,
    y_pred_gradient,
    zero_division=0
)

recall_gradient = recall_score(
    y_test,
    y_pred_gradient,
    zero_division=0
)

f1_gradient = f1_score(
    y_test,
    y_pred_gradient,
    zero_division=0
)

roc_auc_gradient = roc_auc_score(
    y_test,
    y_proba_gradient
)

print("===== GRADIENT BOOSTING =====")
print(f"Accuracy:  {accuracy_gradient:.4f}")
print(f"Precision: {precision_gradient:.4f}")
print(f"Recall:    {recall_gradient:.4f}")
print(f"F1-score:  {f1_gradient:.4f}")
print(f"ROC AUC:   {roc_auc_gradient:.4f}")

## Validação cruzada do Gradient Boosting

In [ ]:
resultado_cv_gradient = cross_validate(
    modelo_gradient_boosting,
    X_temp,
    y_temp,
    cv=cv_estratificado,
    scoring=metricas_cv,
    n_jobs=-1
)

resumo_cv_gradient = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC AUC"
    ],
    "Média": [
        resultado_cv_gradient[
            "test_accuracy"
        ].mean(),
        resultado_cv_gradient[
            "test_precision"
        ].mean(),
        resultado_cv_gradient[
            "test_recall"
        ].mean(),
        resultado_cv_gradient[
            "test_f1"
        ].mean(),
        resultado_cv_gradient[
            "test_roc_auc"
        ].mean()
    ],
    "Desvio": [
        resultado_cv_gradient[
            "test_accuracy"
        ].std(),
        resultado_cv_gradient[
            "test_precision"
        ].std(),
        resultado_cv_gradient[
            "test_recall"
        ].std(),
        resultado_cv_gradient[
            "test_f1"
        ].std(),
        resultado_cv_gradient[
            "test_roc_auc"
        ].std()
    ]
})

resumo_cv_gradient.round(4)

# 74. XGBoost

O XGBoost é uma implementação otimizada de Gradient Boosting.

Ele inclui regularização, processamento eficiente e diferentes controles de complexidade.

Para lidar com o desbalanceamento, utilizaremos `scale_pos_weight`, calculado a partir da proporção entre as classes no conjunto de treinamento.

In [ ]:
quantidade_negativos = int(
    (y_train == 0).sum()
)

quantidade_positivos = int(
    (y_train == 1).sum()
)

scale_pos_weight = (
    quantidade_negativos
    / quantidade_positivos
)

print(
    f"scale_pos_weight: "
    f"{scale_pos_weight:.4f}"
)

In [ ]:
modelo_xgboost = Pipeline([
    (
        "preprocessador",
        preprocessador
    ),
    (
        "modelo",
        XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1
        )
    )
])

modelo_xgboost.fit(
    X_train,
    y_train
)

print("XGBoost treinado com sucesso.")

In [ ]:
y_pred_xgboost = modelo_xgboost.predict(
    X_test
)

y_proba_xgboost = modelo_xgboost.predict_proba(
    X_test
)[:, 1]

accuracy_xgboost = accuracy_score(
    y_test,
    y_pred_xgboost
)

precision_xgboost = precision_score(
    y_test,
    y_pred_xgboost,
    zero_division=0
)

recall_xgboost = recall_score(
    y_test,
    y_pred_xgboost,
    zero_division=0
)

f1_xgboost = f1_score(
    y_test,
    y_pred_xgboost,
    zero_division=0
)

roc_auc_xgboost = roc_auc_score(
    y_test,
    y_proba_xgboost
)

print("===== XGBOOST =====")
print(f"Accuracy:  {accuracy_xgboost:.4f}")
print(f"Precision: {precision_xgboost:.4f}")
print(f"Recall:    {recall_xgboost:.4f}")
print(f"F1-score:  {f1_xgboost:.4f}")
print(f"ROC AUC:   {roc_auc_xgboost:.4f}")

## Validação cruzada do XGBoost

In [ ]:
resultado_cv_xgboost = cross_validate(
    modelo_xgboost,
    X_temp,
    y_temp,
    cv=cv_estratificado,
    scoring=metricas_cv,
    n_jobs=-1
)

resumo_cv_xgboost = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC AUC"
    ],
    "Média": [
        resultado_cv_xgboost[
            "test_accuracy"
        ].mean(),
        resultado_cv_xgboost[
            "test_precision"
        ].mean(),
        resultado_cv_xgboost[
            "test_recall"
        ].mean(),
        resultado_cv_xgboost[
            "test_f1"
        ].mean(),
        resultado_cv_xgboost[
            "test_roc_auc"
        ].mean()
    ],
    "Desvio": [
        resultado_cv_xgboost[
            "test_accuracy"
        ].std(),
        resultado_cv_xgboost[
            "test_precision"
        ].std(),
        resultado_cv_xgboost[
            "test_recall"
        ].std(),
        resultado_cv_xgboost[
            "test_f1"
        ].std(),
        resultado_cv_xgboost[
            "test_roc_auc"
        ].std()
    ]
})

resumo_cv_xgboost.round(4)

# 75. Comparação ampliada dos modelos

Agora incluiremos Gradient Boosting e XGBoost na tabela de teste.

In [ ]:
comparacao_modelos_avancada = pd.concat([
    comparacao_modelos_threshold,
    pd.DataFrame({
        "Modelo": [
            "Gradient Boosting",
            "XGBoost"
        ],
        "Accuracy": [
            accuracy_gradient,
            accuracy_xgboost
        ],
        "Precision": [
            precision_gradient,
            precision_xgboost
        ],
        "Recall": [
            recall_gradient,
            recall_xgboost
        ],
        "F1-score": [
            f1_gradient,
            f1_xgboost
        ],
        "ROC AUC": [
            roc_auc_gradient,
            roc_auc_xgboost
        ]
    })
], ignore_index=True)

comparacao_modelos_avancada.sort_values(
    by="F1-score",
    ascending=False
).round(4)

# 76. Calibração das probabilidades

Um classificador pode ordenar corretamente os clientes por risco e ainda produzir probabilidades pouco confiáveis.

Por exemplo, entre clientes com previsão próxima de 70%, espera-se que aproximadamente 70% apresentem churn.

Vamos analisar:

- Brier Score;
- Log Loss;
- curva de calibração.

Valores menores de Brier Score e Log Loss são melhores.

In [ ]:
probabilidades_modelos = {
    "Regressão Logística": y_proba_logistico,
    "Random Forest regularizado": (
        y_proba_rf_regularizado
    ),
    "Gradient Boosting": y_proba_gradient,
    "XGBoost": y_proba_xgboost,
    "Rede Neural": y_proba_rede
}

resultados_calibracao = []

for nome, probabilidades in (
    probabilidades_modelos.items()
):
    resultados_calibracao.append({
        "Modelo": nome,
        "Brier Score": brier_score_loss(
            y_test,
            probabilidades
        ),
        "Log Loss": log_loss(
            y_test,
            probabilidades
        ),
        "ROC AUC": roc_auc_score(
            y_test,
            probabilidades
        )
    })

df_calibracao = pd.DataFrame(
    resultados_calibracao
).sort_values(
    by="Brier Score"
).reset_index(drop=True)

df_calibracao.round(4)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 7)
)

for nome, probabilidades in (
    probabilidades_modelos.items()
):
    CalibrationDisplay.from_predictions(
        y_test,
        probabilidades,
        n_bins=10,
        strategy="quantile",
        name=nome,
        ax=ax
    )

plt.title(
    "Curvas de calibração dos modelos"
)
plt.show()

# 77. Calibração do XGBoost

Vamos calibrar o XGBoost com dois métodos:

- `sigmoid`;
- `isotonic`.

A calibração será feita com validação cruzada interna.

Depois, compararemos Brier Score, Log Loss e ROC AUC.

In [ ]:
xgboost_calibrado_sigmoid = (
    CalibratedClassifierCV(
        estimator=modelo_xgboost,
        method="sigmoid",
        cv=5,
        n_jobs=-1
    )
)

xgboost_calibrado_sigmoid.fit(
    X_train,
    y_train
)

y_proba_xgb_sigmoid = (
    xgboost_calibrado_sigmoid.predict_proba(
        X_test
    )[:, 1]
)

In [ ]:
xgboost_calibrado_isotonic = (
    CalibratedClassifierCV(
        estimator=modelo_xgboost,
        method="isotonic",
        cv=5,
        n_jobs=-1
    )
)

xgboost_calibrado_isotonic.fit(
    X_train,
    y_train
)

y_proba_xgb_isotonic = (
    xgboost_calibrado_isotonic.predict_proba(
        X_test
    )[:, 1]
)

In [ ]:
comparacao_calibracao_xgb = pd.DataFrame({
    "Modelo": [
        "XGBoost original",
        "XGBoost calibrado — sigmoid",
        "XGBoost calibrado — isotonic"
    ],
    "Brier Score": [
        brier_score_loss(
            y_test,
            y_proba_xgboost
        ),
        brier_score_loss(
            y_test,
            y_proba_xgb_sigmoid
        ),
        brier_score_loss(
            y_test,
            y_proba_xgb_isotonic
        )
    ],
    "Log Loss": [
        log_loss(
            y_test,
            y_proba_xgboost
        ),
        log_loss(
            y_test,
            y_proba_xgb_sigmoid
        ),
        log_loss(
            y_test,
            y_proba_xgb_isotonic
        )
    ],
    "ROC AUC": [
        roc_auc_score(
            y_test,
            y_proba_xgboost
        ),
        roc_auc_score(
            y_test,
            y_proba_xgb_sigmoid
        ),
        roc_auc_score(
            y_test,
            y_proba_xgb_isotonic
        )
    ]
})

comparacao_calibracao_xgb.sort_values(
    by="Brier Score"
).round(4)

## Seleção da melhor versão calibrada

Escolheremos a versão com menor Brier Score.

In [ ]:
indice_melhor_calibracao = (
    comparacao_calibracao_xgb[
        "Brier Score"
    ].idxmin()
)

nome_melhor_calibracao = (
    comparacao_calibracao_xgb.loc[
        indice_melhor_calibracao,
        "Modelo"
    ]
)

if nome_melhor_calibracao == (
    "XGBoost calibrado — sigmoid"
):
    modelo_xgb_final_calibrado = (
        xgboost_calibrado_sigmoid
    )
    y_proba_xgb_final = y_proba_xgb_sigmoid

elif nome_melhor_calibracao == (
    "XGBoost calibrado — isotonic"
):
    modelo_xgb_final_calibrado = (
        xgboost_calibrado_isotonic
    )
    y_proba_xgb_final = y_proba_xgb_isotonic

else:
    modelo_xgb_final_calibrado = modelo_xgboost
    y_proba_xgb_final = y_proba_xgboost

print(
    f"Melhor versão: "
    f"{nome_melhor_calibracao}"
)

# 78. Ajuste de threshold para o XGBoost calibrado

O threshold será escolhido no conjunto de validação.

Primeiro, geraremos as probabilidades da versão calibrada no conjunto de validação.

In [ ]:
if nome_melhor_calibracao == (
    "XGBoost calibrado — sigmoid"
):
    y_proba_xgb_val = (
        xgboost_calibrado_sigmoid.predict_proba(
            X_val
        )[:, 1]
    )

elif nome_melhor_calibracao == (
    "XGBoost calibrado — isotonic"
):
    y_proba_xgb_val = (
        xgboost_calibrado_isotonic.predict_proba(
            X_val
        )[:, 1]
    )

else:
    y_proba_xgb_val = (
        modelo_xgboost.predict_proba(
            X_val
        )[:, 1]
    )

resultados_threshold_xgb_val = avaliar_thresholds(
    y_val,
    y_proba_xgb_val,
    thresholds
)

melhor_linha_threshold_xgb = (
    resultados_threshold_xgb_val.loc[
        resultados_threshold_xgb_val[
            "F1-score"
        ].idxmax()
    ]
)

threshold_xgb_final = float(
    melhor_linha_threshold_xgb["Threshold"]
)

print(
    f"Threshold XGBoost calibrado: "
    f"{threshold_xgb_final:.2f}"
)

print(
    f"F1 na validação: "
    f"{melhor_linha_threshold_xgb['F1-score']:.4f}"
)

In [ ]:
y_pred_xgb_final = (
    y_proba_xgb_final
    >= threshold_xgb_final
).astype(int)

metricas_xgb_final = {
    "Accuracy": accuracy_score(
        y_test,
        y_pred_xgb_final
    ),
    "Precision": precision_score(
        y_test,
        y_pred_xgb_final,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        y_pred_xgb_final,
        zero_division=0
    ),
    "F1-score": f1_score(
        y_test,
        y_pred_xgb_final,
        zero_division=0
    ),
    "ROC AUC": roc_auc_score(
        y_test,
        y_proba_xgb_final
    ),
    "Brier Score": brier_score_loss(
        y_test,
        y_proba_xgb_final
    )
}

pd.DataFrame(
    [metricas_xgb_final]
).round(4)

# 79. Explicabilidade com SHAP

SHAP atribui uma contribuição a cada variável para explicar uma previsão.

Valores SHAP positivos empurram a previsão em direção a maior risco de churn.

Valores negativos empurram a previsão em direção a menor risco.

A análise será feita sobre o XGBoost original, pois ele é diretamente compatível com explicadores de árvores.

In [ ]:
preprocessador_xgb = (
    modelo_xgboost.named_steps[
        "preprocessador"
    ]
)

xgb_estimador = modelo_xgboost.named_steps[
    "modelo"
]

X_train_xgb_processado = (
    preprocessador_xgb.transform(
        X_train
    )
)

X_test_xgb_processado = (
    preprocessador_xgb.transform(
        X_test
    )
)

nomes_features_xgb = (
    preprocessador_xgb
    .get_feature_names_out()
)

X_test_xgb_df = pd.DataFrame(
    X_test_xgb_processado,
    columns=nomes_features_xgb,
    index=X_test.index
)

print(
    "Shape dos dados processados:",
    X_test_xgb_df.shape
)

In [ ]:
explainer_xgb = shap.TreeExplainer(
    xgb_estimador
)

shap_values_xgb = explainer_xgb(
    X_test_xgb_df
)

print(
    "Valores SHAP calculados com sucesso."
)

## Importância global das variáveis

O gráfico abaixo mostra quais features tiveram maior influência média nas previsões.

In [ ]:
shap.plots.bar(
    shap_values_xgb,
    max_display=15
)

## Distribuição das contribuições

O beeswarm mostra:

- importância;
- direção do efeito;
- intensidade do valor da feature.

In [ ]:
shap.plots.beeswarm(
    shap_values_xgb,
    max_display=15
)

## Explicação de uma previsão individual

O gráfico waterfall mostra como cada variável contribuiu para a previsão de um cliente.

In [ ]:
indice_exemplo_shap = 0

shap.plots.waterfall(
    shap_values_xgb[
        indice_exemplo_shap
    ],
    max_display=15
)

## Dados originais do cliente explicado

In [ ]:
cliente_explicado = X_test.iloc[
    [indice_exemplo_shap]
]

probabilidade_cliente = (
    y_proba_xgboost[
        indice_exemplo_shap
    ]
)

classe_real_cliente = int(
    y_test.iloc[indice_exemplo_shap]
)

display(cliente_explicado)

print(
    f"Probabilidade prevista: "
    f"{probabilidade_cliente:.4f}"
)

print(
    f"Classe real: "
    f"{classe_real_cliente}"
)

# 80. Simulação financeira de retenção

Um modelo não deve ser avaliado apenas por métricas técnicas.

Precisamos estimar se a campanha de retenção gera valor econômico.

A simulação utilizará premissas ajustáveis:

- custo para abordar cada cliente;
- valor médio preservado quando um churn é evitado;
- taxa de sucesso da ação de retenção;
- diferentes thresholds.

A simulação é didática e não representa valores reais da empresa.

In [ ]:
custo_por_abordagem = 25.00
valor_cliente_preservado = 600.00
taxa_sucesso_retencao = 0.30

print(
    f"Custo por abordagem: "
    f"R$ {custo_por_abordagem:.2f}"
)

print(
    f"Valor preservado por cliente retido: "
    f"R$ {valor_cliente_preservado:.2f}"
)

print(
    f"Taxa de sucesso da retenção: "
    f"{taxa_sucesso_retencao:.0%}"
)

## Função de simulação financeira

Para cada threshold:

1. selecionamos os clientes previstos como churn;
2. calculamos o custo total das abordagens;
3. identificamos quantos churns reais foram alcançados;
4. estimamos quantos seriam retidos;
5. calculamos receita preservada;
6. calculamos resultado líquido;
7. calculamos retorno sobre investimento.

In [ ]:
def simular_campanha_retencao(
    y_real,
    probabilidades,
    thresholds,
    custo_abordagem,
    valor_preservado,
    taxa_sucesso
):
    resultados = []

    for threshold in thresholds:
        selecionados = (
            probabilidades >= threshold
        )

        quantidade_abordada = int(
            selecionados.sum()
        )

        churns_reais_alcancados = int(
            (
                selecionados
                & (np.asarray(y_real) == 1)
            ).sum()
        )

        clientes_retidos_estimados = (
            churns_reais_alcancados
            * taxa_sucesso
        )

        custo_total = (
            quantidade_abordada
            * custo_abordagem
        )

        receita_preservada = (
            clientes_retidos_estimados
            * valor_preservado
        )

        resultado_liquido = (
            receita_preservada
            - custo_total
        )

        roi = (
            resultado_liquido
            / custo_total
            if custo_total > 0
            else 0
        )

        resultados.append({
            "Threshold": threshold,
            "Clientes abordados": (
                quantidade_abordada
            ),
            "Churns reais alcançados": (
                churns_reais_alcancados
            ),
            "Retidos estimados": (
                clientes_retidos_estimados
            ),
            "Custo total": custo_total,
            "Receita preservada": (
                receita_preservada
            ),
            "Resultado líquido": (
                resultado_liquido
            ),
            "ROI": roi
        })

    return pd.DataFrame(resultados)

In [ ]:
thresholds_financeiros = np.arange(
    0.10,
    0.91,
    0.05
)

simulacao_financeira = (
    simular_campanha_retencao(
        y_real=y_test,
        probabilidades=y_proba_xgb_final,
        thresholds=thresholds_financeiros,
        custo_abordagem=custo_por_abordagem,
        valor_preservado=valor_cliente_preservado,
        taxa_sucesso=taxa_sucesso_retencao
    )
)

simulacao_financeira.round(2)

## Threshold com maior resultado líquido

In [ ]:
melhor_cenario_financeiro = (
    simulacao_financeira.loc[
        simulacao_financeira[
            "Resultado líquido"
        ].idxmax()
    ]
)

print(
    "===== MELHOR CENÁRIO FINANCEIRO ====="
)

print(
    f"Threshold: "
    f"{melhor_cenario_financeiro['Threshold']:.2f}"
)

print(
    f"Clientes abordados: "
    f"{int(melhor_cenario_financeiro['Clientes abordados'])}"
)

print(
    f"Churns reais alcançados: "
    f"{int(melhor_cenario_financeiro['Churns reais alcançados'])}"
)

print(
    f"Clientes retidos estimados: "
    f"{melhor_cenario_financeiro['Retidos estimados']:.1f}"
)

print(
    f"Custo total: R$ "
    f"{melhor_cenario_financeiro['Custo total']:.2f}"
)

print(
    f"Receita preservada: R$ "
    f"{melhor_cenario_financeiro['Receita preservada']:.2f}"
)

print(
    f"Resultado líquido: R$ "
    f"{melhor_cenario_financeiro['Resultado líquido']:.2f}"
)

print(
    f"ROI: "
    f"{melhor_cenario_financeiro['ROI']:.2%}"
)

In [ ]:
plt.figure(
    figsize=(11, 6)
)

plt.plot(
    simulacao_financeira["Threshold"],
    simulacao_financeira["Resultado líquido"],
    marker="o"
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Threshold")
plt.ylabel("Resultado líquido estimado (R$)")
plt.title(
    "Resultado financeiro por threshold"
)
plt.grid(True)
plt.show()

# 81. Análise de sensibilidade financeira

Como as premissas são incertas, vamos testar diferentes taxas de sucesso da campanha.

In [ ]:
taxas_sucesso = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

resultados_sensibilidade = []

for taxa in taxas_sucesso:
    simulacao_taxa = simular_campanha_retencao(
        y_real=y_test,
        probabilidades=y_proba_xgb_final,
        thresholds=thresholds_financeiros,
        custo_abordagem=custo_por_abordagem,
        valor_preservado=valor_cliente_preservado,
        taxa_sucesso=taxa
    )

    melhor_taxa = simulacao_taxa.loc[
        simulacao_taxa[
            "Resultado líquido"
        ].idxmax()
    ]

    resultados_sensibilidade.append({
        "Taxa de sucesso": taxa,
        "Melhor threshold": (
            melhor_taxa["Threshold"]
        ),
        "Resultado líquido máximo": (
            melhor_taxa[
                "Resultado líquido"
            ]
        ),
        "ROI máximo": melhor_taxa["ROI"]
    })

df_sensibilidade = pd.DataFrame(
    resultados_sensibilidade
)

df_sensibilidade.round(2)

# 82. Salvamento dos novos resultados

Vamos salvar:

- comparação avançada;
- resultados da validação cruzada;
- calibração;
- simulação financeira;
- XGBoost calibrado.

In [ ]:
pasta_artefatos_avancados = Path(
    "artefatos_projeto_02_avancado"
)

pasta_artefatos_avancados.mkdir(
    exist_ok=True
)

comparacao_modelos_avancada.to_csv(
    pasta_artefatos_avancados
    / "comparacao_modelos_avancada.csv",
    index=False
)

resultados_cv.to_csv(
    pasta_artefatos_avancados
    / "validacao_cruzada.csv",
    index=False
)

df_calibracao.to_csv(
    pasta_artefatos_avancados
    / "calibracao_modelos.csv",
    index=False
)

simulacao_financeira.to_csv(
    pasta_artefatos_avancados
    / "simulacao_financeira.csv",
    index=False
)

joblib.dump(
    modelo_xgb_final_calibrado,
    pasta_artefatos_avancados
    / "xgboost_calibrado.pkl"
)

with (
    pasta_artefatos_avancados
    / "threshold_xgboost.json"
).open(
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        {
            "threshold": threshold_xgb_final,
            "metodo_calibracao": (
                nome_melhor_calibracao
            )
        },
        arquivo,
        ensure_ascii=False,
        indent=2
    )

print("Artefatos avançados salvos.")

# Conclusão avançada do Projeto 02

O Projeto 02 passou a contemplar um fluxo mais completo de classificação e decisão de negócio.

Além dos modelos iniciais, foram adicionados:

1. validação cruzada estratificada;
2. Gradient Boosting;
3. XGBoost;
4. comparação de estabilidade entre folds;
5. análise da calibração das probabilidades;
6. calibração do XGBoost com Sigmoid e Isotonic;
7. ajuste de threshold após calibração;
8. explicabilidade global e individual com SHAP;
9. simulação financeira de campanhas de retenção;
10. análise de sensibilidade das premissas financeiras;
11. salvamento dos artefatos avançados.

## Aprendizados adicionais

A validação cruzada permite verificar se o desempenho é consistente em diferentes divisões dos dados.

Gradient Boosting e XGBoost podem capturar relações mais complexas do que modelos lineares.

Uma probabilidade com boa ROC AUC não é necessariamente bem calibrada. Por isso, Brier Score, Log Loss e curvas de calibração complementam a avaliação.

SHAP ajuda a explicar quais variáveis contribuíram para aumentar ou reduzir o risco previsto de cada cliente.

A simulação financeira mostra que o melhor threshold técnico pode não ser o melhor threshold econômico.

A decisão final deve combinar:

- capacidade preditiva;
- estabilidade;
- qualidade das probabilidades;
- explicabilidade;
- custo das ações;
- valor financeiro preservado.

## Limitações

- os valores financeiros usados são premissas hipotéticas;
- a taxa de sucesso da retenção não foi observada em dados reais;
- as explicações SHAP representam o comportamento do modelo, não causalidade;
- o conjunto de teste ainda é usado com finalidade didática;
- uma implantação real exigiria monitoramento, reavaliação periódica e testes controlados.

## Próximo passo

Com os experimentos concluídos, o próximo passo é criar a interface Streamlit utilizando:

- o modelo calibrado;
- o threshold escolhido;
- a faixa de risco;
- explicações SHAP;
- uma estimativa financeira da ação de retenção.